In [1]:
import pandas as pd 
import ast
import numpy as np

In [2]:
%cd Thesis-FOS-BinaryClass-WSD/Ensemble Model/

c:\Users\Miguel\Documents\Project Source Files\IT Work\School\Thesis\Thesis-FOS-BinaryClass-WSD\Ensemble Model


c:\Users\Miguel\Documents\Project Source Files\IT Work\School\Thesis\.venv\Lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


# Load Models

## Sentence Transformer -- Sentence Embeddings


In [3]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('sentence-transformers/LaBSE')

c:\Users\Miguel\Documents\Project Source Files\IT Work\School\Thesis\.venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


# Processing

## Load Dataset

In [4]:
df_test = pd.read_excel('../../Dataset/Test_Set.xlsx')
df_train= pd.read_excel('../../Dataset/Train_Set.xlsx')
display(df_test.head(1),df_test.shape,df_train.head(1),df_train.shape)

,FOS,Word Sense,Verb,Non-Literal Usage,Literal Usage
0,lunod patay,dihard,"['lunod', 'patay']",Si Maria lunod patay sa pagpanalipod sa iyang ...,Ang balita miingon nga dunay usa ka tawo nga l...


(161, 5)

,FOS,Word Sense,Verb,Non-Literal Usage,Literal Usage
0,hangyo lubo,usa ka hangyo nga gihimo sa ingon nga paagi ng...,"['hangyo', 'lubo']","Si Ana hangyo lubo, maong nakuha niya ang iyan...",Ang bata naghangyo og lubo sa iyang amahan aro...


(644, 5)

## Sentence Transformer

Removing the label column from the dataset for clarity

In [5]:
# df_train.drop(columns=['Is FOS'], inplace=True)
df = df_train.copy()

In [6]:
# df['Word Sense'] = df['Word Sense']
df.head(3)

,FOS,Word Sense,Verb,Non-Literal Usage,Literal Usage
0,hangyo lubo,usa ka hangyo nga gihimo sa ingon nga paagi ng...,"['hangyo', 'lubo']","Si Ana hangyo lubo, maong nakuha niya ang iyan...",Ang bata naghangyo og lubo sa iyang amahan aro...
1,bukhad og palad,mahinatagon Synonyms: manggihatagon,['bukhad'],"Si Juan bukhad og palad, bisan sa iyang kaliso...",Si Manang Rosa mihangyo kang Simoun nga bukhad...
2,hatod og patay,sa pagtambong sa usa ka paglubong o seremonya ...,"['hatod', 'patay']",Si Maria hatod og patay sa iyang silingan nga ...,Nakit-an namo si Juan nga naghatod og patay na...


In [7]:
def GetEmbeddings(words:str) -> np.ndarray:
    return embedding_model.encode(words)

def ComputeSimilarity(embedding1,embedding2):
    return embedding_model.similarity(embedding1, embedding2)

### Generate Embeddings

> Generating the embeddings from the verb could be done another way than simply entering the list into the embeddings model

In [9]:
df['Sentence Embeddings'] = df['Word Sense'].apply(GetEmbeddings)
df['Verb Embeddings'] = df['Verb'].apply(GetEmbeddings)
df['Literal Usage Embeddings'] = df['Literal Usage'].apply(GetEmbeddings)
df['Non-Literal Usage Embeddings'] = df['Non-Literal Usage'].apply(GetEmbeddings)
df.head(3)

,FOS,Word Sense,Verb,Non-Literal Usage,Literal Usage,Sentence Embeddings,Verb Embeddings,Literal Usage Embeddings,Non-Literal Usage Embeddings
0,hangyo lubo,usa ka hangyo nga gihimo sa ingon nga paagi ng...,"['hangyo', 'lubo']","Si Ana hangyo lubo, maong nakuha niya ang iyan...",Ang bata naghangyo og lubo sa iyang amahan aro...,"[0.013299048, 0.0050963946, -0.025889978, -0.0...","[-0.024435937, 0.0056826, -0.016659597, -0.020...","[-0.035629567, -0.05257719, 0.04578231, 0.0097...","[-0.05293895, -0.029694155, 0.008109281, 0.071..."
1,bukhad og palad,mahinatagon Synonyms: manggihatagon,['bukhad'],"Si Juan bukhad og palad, bisan sa iyang kaliso...",Si Manang Rosa mihangyo kang Simoun nga bukhad...,"[0.018145593, -0.041875407, 0.058073983, 0.008...","[-0.005556708, -0.014158582, -0.043383628, -0....","[-0.009618969, -0.009924677, 0.026966063, 0.03...","[-0.04525887, -0.049815923, -0.0160535, 0.0233..."
2,hatod og patay,sa pagtambong sa usa ka paglubong o seremonya ...,"['hatod', 'patay']",Si Maria hatod og patay sa iyang silingan nga ...,Nakit-an namo si Juan nga naghatod og patay na...,"[-0.007992692, 0.03143906, -0.028817644, -0.05...","[-0.051542707, 0.008255355, -0.034022376, 0.01...","[0.001030946, 0.030172313, -0.038251866, -0.00...","[-0.0139160985, 0.033600938, 0.02302225, -0.02..."


### Compute Similarity Scores

In [11]:
df[:2].apply(lambda row: print(ComputeSimilarity(row['Sentence Embeddings'], row['Literal Usage Embeddings'])),axis=1)

tensor([[0.2839]])
tensor([[0.2069]])


0    None
1    None
dtype: object

In [12]:
df['Literal Similarity'] = df.apply(lambda row: ComputeSimilarity(row['Sentence Embeddings'], row['Literal Usage Embeddings']),axis=1)
df['Non-literal Similarity'] = df.apply(lambda row: ComputeSimilarity(row['Sentence Embeddings'], row['Non-Literal Usage Embeddings']),axis=1)

In [13]:
df.head(3)

,FOS,Word Sense,Verb,Non-Literal Usage,Literal Usage,Sentence Embeddings,Verb Embeddings,Literal Usage Embeddings,Non-Literal Usage Embeddings,Literal Similarity,Non-literal Similarity
0,hangyo lubo,usa ka hangyo nga gihimo sa ingon nga paagi ng...,"['hangyo', 'lubo']","Si Ana hangyo lubo, maong nakuha niya ang iyan...",Ang bata naghangyo og lubo sa iyang amahan aro...,"[0.013299048, 0.0050963946, -0.025889978, -0.0...","[-0.024435937, 0.0056826, -0.016659597, -0.020...","[-0.035629567, -0.05257719, 0.04578231, 0.0097...","[-0.05293895, -0.029694155, 0.008109281, 0.071...",[[tensor(0.2839)]],[[tensor(0.2586)]]
1,bukhad og palad,mahinatagon Synonyms: manggihatagon,['bukhad'],"Si Juan bukhad og palad, bisan sa iyang kaliso...",Si Manang Rosa mihangyo kang Simoun nga bukhad...,"[0.018145593, -0.041875407, 0.058073983, 0.008...","[-0.005556708, -0.014158582, -0.043383628, -0....","[-0.009618969, -0.009924677, 0.026966063, 0.03...","[-0.04525887, -0.049815923, -0.0160535, 0.0233...",[[tensor(0.2069)]],[[tensor(0.1078)]]
2,hatod og patay,sa pagtambong sa usa ka paglubong o seremonya ...,"['hatod', 'patay']",Si Maria hatod og patay sa iyang silingan nga ...,Nakit-an namo si Juan nga naghatod og patay na...,"[-0.007992692, 0.03143906, -0.028817644, -0.05...","[-0.051542707, 0.008255355, -0.034022376, 0.01...","[0.001030946, 0.030172313, -0.038251866, -0.00...","[-0.0139160985, 0.033600938, 0.02302225, -0.02...",[[tensor(0.2050)]],[[tensor(0.2985)]]


## PCA-Guided K-means


## Classification Models

# Output